# ML Practical 3: Evaluation of multiple models

## The task.

Task: Predict whether a person makes over $50k per year from census data known about them.

Data set from the paper: Kohavi, Ron. "Scaling up the accuracy of Naive-Bayes classifiers: a decision-tree hybrid." KDD. Vol. 96. 1996.
Data URL: We will be using modified versions of the publically avaliable data. Please download the data from the URLs provided.

In [2]:
import pandas as pd

# Output feature table
output_feature = pd.DataFrame({
    "Feature": ["salary"],
    "Type": ["categorical"],
    "Values": [">50K, <=50K"]
})

output_feature

,Feature,Type,Values
0,salary,categorical,">50K, <=50K"


In [3]:
# Input feature table
input_features = pd.DataFrame({
    "Feature": [
        "age",
        "workclass",
        "fnlwgt",
        "education",
        "education-num",
        "marital-status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
        "native-country"
    ],
    "Type": [
        "continuous",
        "categorical",
        "continuous",
        "categorical",
        "continuous",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "continuous",
        "continuous",
        "continuous",
        "categorical"
    ],
    "Values": [
        "",
        "Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked",
        "",
        "Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool",
        "",
        "Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse",
        "Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces",
        "Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried",
        "White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black",
        "Female, Male",
        "",
        "",
        "",
        "United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands"
    ]
})

input_features

,Feature,Type,Values
0,age,continuous,
1,workclass,categorical,"Private, Self-emp-not-inc, Self-emp-inc, Feder..."
2,fnlwgt,continuous,
3,education,categorical,"Bachelors, Some-college, 11th, HS-grad, Prof-s..."
4,education-num,continuous,
5,marital-status,categorical,"Married-civ-spouse, Divorced, Never-married, S..."
6,occupation,categorical,"Tech-support, Craft-repair, Other-service, Sal..."
7,relationship,categorical,"Wife, Own-child, Husband, Not-in-family, Other..."
8,race,categorical,"White, Asian-Pac-Islander, Amer-Indian-Eskimo,..."
9,sex,categorical,"Female, Male"


In [4]:
all_features = pd.concat([
    input_features.assign(Role="Input"),
    output_feature.assign(Role="Output")
], ignore_index=True)

all_features = all_features[["Role", "Feature", "Type", "Values"]]
all_features

,Role,Feature,Type,Values
0,Input,age,continuous,
1,Input,workclass,categorical,"Private, Self-emp-not-inc, Self-emp-inc, Feder..."
2,Input,fnlwgt,continuous,
3,Input,education,categorical,"Bachelors, Some-college, 11th, HS-grad, Prof-s..."
4,Input,education-num,continuous,
5,Input,marital-status,categorical,"Married-civ-spouse, Divorced, Never-married, S..."
6,Input,occupation,categorical,"Tech-support, Craft-repair, Other-service, Sal..."
7,Input,relationship,categorical,"Wife, Own-child, Husband, Not-in-family, Other..."
8,Input,race,categorical,"White, Asian-Pac-Islander, Amer-Indian-Eskimo,..."
9,Input,sex,categorical,"Female, Male"


# To help you along some of the basic data preparation has been done for you.
Read the code. Understand what has been done.

In [ ]:
# Some basic imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

# Read the data into a pandas DataFrame
data = pd.read_csv('https://drive.google.com/uc?export=download&id=1lBiNrYBk5KdfBllyjuELgRwYaK4yT2z9', header = 0, names = ['age','workclass','fnlwgt','education','education-num','matrial-status','occupation','relationship','race','sex','captial-gain','captial-loss','hours-per-week','salary'])

In [ ]:
# Define our input features and our output feature
# Call our input features X and our output feature y (the sklearn standard)
# Note that we have categorical features.
X = data.drop( columns = 'salary' )
y = data.salary

In [ ]:
# Now we need to encode our output feature to be an integer 0 or 1.
# This is because we have a binary classification problem and in order to use sklearn's
# built-in evaluation measures we need to have one class defined as 1 (target) and one as 0 (non-target).

# We could do this by using the LabelEncoder from sklearn. The LabelEncoder will convert n-distinct values
# to 0,..,n-1 values in this case giving us what we want. We assume that our training set contains both
# labels and that this mapping will be valid. However, we have no control
# over which value is represented by 1 and which is represented by 0.
# Therefore it is easier (in terms of subsequent interpretation) to do this
# manually. Recall the problem, we want our target variable (1) to be '>50k'

# To do this (your variable y is a pandas.Series object, use the replace method):
# 1) update all values '<=50K' within y to equal 0
# 2) update all values '>50K' within y to equal 1

y.replace(to_replace = ' <=50K', value = 0, inplace = True)
y.replace(to_replace = ' >50K', value = 1, inplace = True)

<ipython-input-3-e754d1361b9a>:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace(to_replace = ' >50K', value = 1, inplace = True)


In [ ]:
# The baseline classifier for you to use
lr_model = Pipeline([
    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
    ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine after onehot)
    ('model',LogisticRegression(solver = 'liblinear') )
    ])



# Your turn. See the instructions in the slide deck...

In [ ]:
def fit_model(X, y):
    kf_inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    # For the model class RF
    rf_model = Pipeline( [
        ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
        ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
        ('model',RandomForestClassifier() )
    ] )
    search = {'model__n_estimators':[10,100]}
    rf_cv_model = GridSearchCV(rf_model, search, cv = kf_inner, refit = True)
    rf_cv_model.fit(X, y)
    rf_best_model = rf_cv_model.best_estimator_

    # For the model class SVM
    svm_model = Pipeline( [
        ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output = False)),  # will automatically pick string columns (could have specified)
        ('standardize', StandardScaler()), # will convert everything (can't specify which columns but all columns are fine)
        ('model',SVC() )
    ] )
    search = {'model__C':[1,10],'model__kernel':['rbf','linear']}
    svm_cv_model = GridSearchCV(svm_model, search, cv = kf_inner, refit = True)
    svm_cv_model.fit(X, y)
    svm_best_model = svm_cv_model.best_estimator_

    # The baseline
    # we take the mean as best_estimator_ is the mean of the CV folds
    # but cross_val_score returns each folds score (see the documentation)
    lr_score = np.mean(cross_val_score(lr_model,X, y, cv = kf_inner))
    # refit it to all data in case we select it
    # (for computational reasons we probably should only do this if required, additionally we shouldn't call refit = True for
    #   models above but only refit the model that we end up selecting)
    lr_model.fit(X,y)

    model_set = [rf_best_model, svm_best_model, lr_model]
    score_set = [rf_cv_model.best_score_,  svm_cv_model.best_score_, lr_score]

    # Keep the best scoring model
    best_model = model_set[ np.argmax(score_set) ]
    return best_model

# Define the CV strategy
kf_outter = StratifiedKFold(n_splits=3, shuffle=True, random_state=13111985)
results = []
# For each fold
for train_index, test_index in kf_outter.split(X,y):
    X_train = X.iloc[train_index]
    y_train = y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test = y.iloc[test_index]

    # Fit & score the model
    best_model = fit_model(X_train, y_train)
    score = best_model.score(X_test, y_test)
    results.append(score)

# Print the result
print(np.mean(results))

# Train deployment version
deploy = fit_model(X, y)

0.8262816847722508
